<a href="https://colab.research.google.com/github/ollihansen90/Mathe-SH/blob/main/Schiffeversenken_Termin4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚢 Schiffe Versenken - Termin 4
## Fragen?
Solltet ihr Fragen zum Code oder Probleme mit Colab haben, schickt uns gerne eine Mail:

*   h.hansen@uni-luebeck.de
*   mika.kohlhammer@student.uni-luebeck.de
*   gun.ingwersen@student.uni-luebeck.de

## 🚢 Das eigene Spiel 💻

In den letzten Terminen haben wir bereits viele wichtige Bausteine kennengelernt:  
Wir haben mit Listen gearbeitet, Funktionen geschrieben und gelernt, wie man Spielfelder darstellen kann. 🧠✨

Heute kommt der spannendste Teil:  
Wir werden Schritt für Schritt unser eigenes **Schiffe-versenken-Spiel** programmieren! 🎯

Dabei werden wir:

- ein Spielfeld erstellen 🟦  
- Schiffe platzieren 🚢  
- Eingaben der Spielenden verarbeiten ⌨️  
- Treffer und Fehlversuche erkennen 💥🌊  
- und am Ende ein vollständig spielbares Spiel haben! 🎉

Der folgende Codeblock enthält unsere Hilfsfunktionen, die wir bereits in den letzten Wochen zusammenprogrammiert haben.

In [ ]:
# utils.py (möglicherweise als eigene Datei zum Importieren, siehe bspw. https://github.com/ollihansen90/Mathe-SH/blob/main/SudokuSolver.ipynb)
def reset_spielfeld():
    # Erstellt ein leeres Spielfeld
    return [[0 for i in range(10)] for j in range(10)]

def male_feld(feld):
    # Zeichnet das Spielfeld auf der Konsole
    print("  │ 1 2 3 4 5 6 7 8 9 0 │  ")
    print("──┼─────────────────────┼─")
    alphabet = "abcdefghijklmnopqrstuvwxyz"
    for i, zeile in enumerate(feld):
        string = alphabet[i] + " │ "
        for zeichen in zeile:
            string = string+"~sxtz"[zeichen] + " "
        string = string + "│"
        print(string)
    print("──┼─────────────────────┼─")

def setze_schiff(feld, pos, dir, l):
    # Setzt Schiff in das Spielfeld
    #   pos: Koordinaten, [int, int]
    #   dir: Ausrichtung, 0 horizontal (waagerecht), 1 vertikal (senkrecht), int
    #   l: Länge, int
    for i in range(l):
        if dir==0:
            feld[pos[0]][pos[1]+i] = 1
        else:
            feld[pos[0]+i][pos[1]] = 1

    return feld

def padding(feld):
    # Rahmen mit Nullen
    #   feld: Spielfeld, Liste von Listen
    output = [[0 for i in range(12)] for j in range(12)]
    for i in range(10):
        for j in range(10):
            output[i+1][j+1] = feld[i][j]
    return output

def erlaubt(feld, dir, l):
    # Erstellt für gegebenes Feld die möglichen Startposition eines Schiffes
    #   feld: Spielfeld, Liste von Listen mit int
    #   dir: Ausrichtung, 0 horizontal (waagerecht), 1 vertikal (senkrecht), int
    #   l: Länge, int
    hilfsfeld = [[1 for i in range(10)] for j in range(10)]
    p_feld = padding(feld)
    # Rand
    if dir==0:
        for zeile in range(10):
            for spalte in range(10-l+1, 10):
                hilfsfeld[zeile][spalte] = 0
    else:
        for zeile in range(10-l+1, 10):
            for spalte in range(10):
                hilfsfeld[zeile][spalte] = 0
    # Schiff-Nachbarn
    for zeile in range(10):
        for spalte in range(10): # Tile zeile, spalte
            if hilfsfeld[zeile][spalte] == 0:
                continue
            summe = 0
            for kx in range(3): # k für Kernel
                for ky in range(3):
                    summe = summe + p_feld[zeile+kx][spalte+ky]

            if summe>0:
                hilfsfeld[zeile][spalte] = 2

    # Crash
    for zeile in range(10):
        for spalte in range(10): # Tile zeile, spalte
            if hilfsfeld[zeile][spalte]==2:
                for i in range(1, l):
                    if dir==1 and zeile-i>=0:
                        hilfsfeld[zeile-i][spalte] = 3
                    elif dir==0 and spalte-i>=0:
                        hilfsfeld[zeile][spalte-i] = 3

    output = []
    for x in range(len(hilfsfeld)):
        for y in range(len(hilfsfeld[x])):
            if hilfsfeld[x][y] == 1:
                output.append([x, y])

    return output
import random

def generiere_spielfeld(schiffe=[5,4,3,3,2]):
    # Erstellt ein leeres Spielfeld und setzt Schiffe rein
    #   schiffe: Liste von Schifflängen
    spielfeld = reset_spielfeld()
    for l in schiffe:
        dir = random.randint(0,1)
        erlaubtliste = erlaubt(spielfeld, dir, l)
        pos = erlaubtliste[random.randint(0,len(erlaubtliste)-1)]
        spielfeld = setze_schiff(spielfeld, pos, dir, l)
    return spielfeld

spielfeld = generiere_spielfeld()
male_feld(spielfeld)

## ⌨️ Schritt 1: Eingaben der Spielenden verarbeiten

Damit unser Spiel funktioniert, müssen die Spielenden Felder auswählen können.  
Dazu brauchen wir zwei wichtige Funktionen:  

- `gueltig(eingabe)` → überprüft, ob eine Eingabe erlaubt ist ✅  
- `eingabe()` → fragt eine Eingabe ab und wandelt sie in Koordinaten um 🎯  

### 🧩 Wie funktioniert die Eingabe?

Unser Spielfeld verwendet folgende Koordinaten:

- **Buchstaben a–j** für die Zeilen  
- **Zahlen 1–0** für die Spalten  

Beispiele für gültige Eingaben sind:

- `a1`
- `c5`
- `j0`

Ungültige Eingaben wären zum Beispiel:

- `aa` ❌  
- `z9` ❌  
- `a12` ❌  
- `5a` ❌  

---

### 🛠️ Eure Aufgaben

#### 1. Funktion `gueltig(eingabe)`
Diese Funktion soll überprüfen, ob die Eingabe gültig ist.

Sie soll:

- `True` zurückgeben, wenn die Eingabe gültig ist ✅  
- `False` zurückgeben, wenn die Eingabe ungültig ist ❌  

Überlegt euch:

- Wie lang muss die Eingabe sein?  
- Welche Buchstaben sind erlaubt?  
- Welche Zahlen sind erlaubt?  

---

#### 2. Funktion `eingabe()`
Diese Funktion soll:

- die Spielenden nach einer Eingabe fragen ⌨️  
- die Eingabe überprüfen  
- und die passende Position als Liste zurückgeben, zum Beispiel:

```python
[0, 0]   # entspricht a1
[2, 4]   # entspricht c5
```

In [ ]:
def gueltig(eingabe):
    return True

def eingabe():
    eingabe = ""
    output = []
    return output

print(eingabe())

## 🏁 Schritt 2: Überprüfen, ob das Spiel beendet ist

Jetzt brauchen wir eine Möglichkeit, um festzustellen, wann das Spiel vorbei ist. 🎯  

Das Spiel ist beendet, wenn **alle Schiffe vollständig gefunden wurden**. 🚢💥

Das bedeutet:  
Alle Felder, auf denen sich ein Schiff befindet, müssen auch im Spielfeld der Spielenden als Treffer markiert sein.

---

### 🧩 Was bedeuten die Variablen?

Ihr arbeitet mit zwei Spielfeldern:

- `loesung` → enthält die echten Positionen der Schiffe 🚢  
- `spielfeld` → enthält die bisherigen Versuche der Spielenden 🎯  

In beiden Spielfeldern gilt:

- `0` → unbekannt / noch nicht getroffen 🌊  
- `1` → Treffer 💥  
- `2` → kein Schiff vorhanden ❌  

---

### 🛠️ Eure Aufgabe

Schreibt die Funktion:

```python
spiel_ende(spielfeld, loesung)
```
Diese Funktion soll:
- alle Felder überprüfen 🔍
- feststellen, ob noch Schiffsteile übrig sind, die nicht getroffen wurden
- und dann zurückgeben:

```python
True   # wenn alle Schiffe gefunden wurden 🎉
False  # wenn noch Schiffe übrig sind 🚢
```

Wenn die Funktion fertig ist, kann das Spiel später automatisch erkennen, wann es vorbei ist! 🏁

In [ ]:
def spiel_ende(spielfeld, loesung):
    # TODO
    return True

loesung = generiere_spielfeld()
spielfeld = reset_spielfeld()



## 🎮 Schritt 3: Das fertige Spiel zusammensetzen

Jetzt kommt der letzte Schritt: Wir verbinden alles zu einem funktionierenden Spiel! 🚀  

Die Schleife läuft so lange, bis alle Schiffe gefunden wurden. In jedem Durchlauf passiert Folgendes:

1. Die spielende Person gibt ein Feld ein ⌨️  
2. Dieses Feld wird überprüft 🔍  
3. Das Spielfeld wird entsprechend aktualisiert 🎯  

---

### 🧩 Aufgabe

Wir erhalten die Koordinaten aus der Funktion `eingabe()`. Jetzt müsssen wir überprüfen:

- Befindet sich an dieser Stelle ein Schiff? 🚢  
  → Dann ist es ein **Treffer** 💥  

- Oder befindet sich dort kein Schiff? 🌊  
  → Dann ist es ein **Fehlversuch** ❌  

Wir speichern das Ergebnis im `spielfeld`:

- `1` für Treffer 💥  
- `2` für Fehlversuch ❌  

---

### 💡 Überlegt euch:

- Wie greift ihr auf die richtige Zeile und Spalte zu?  
- Wie könnt ihr prüfen, ob dort in der `loesung` eine `1` steht?  
- Wie verändert ihr den Wert im `spielfeld`?

---

### 🎉 Ziel

Wenn ihr fertig seid, habt ihr ein vollständig funktionierendes "Schiffe versenken"-Spiel!

Die Spielenden können dann:

- Felder auswählen 🎯  
- Treffer erzielen 💥  
- und alle Schiffe finden, um das Spiel zu gewinnen 🏆  

Viel Erfolg beim Fertigstellen eures Spiels! 🚢

In [ ]:
loesung = generiere_spielfeld()
spielfeld = reset_spielfeld()

while not spiel_ende(spielfeld, loesung):
    koordinaten = eingabe()
    # TODO